# GRU Book Genre Classification

Run all cells from top to bottom to load the improved 12-class dataset, split it correctly, train the hybrid GRU model, save a checkpoint, and print the final metrics.

## 1. Load Dataset

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras import layers, models, regularizers

RANDOM_STATE = 42
DATASET_NAME = "BooksClassifier_dataset_reduced_12class.csv"
tf.keras.utils.set_random_seed(RANDOM_STATE)

candidate_paths = [
    Path.cwd() / DATASET_NAME,
    Path.cwd().parent / DATASET_NAME,
]

DATA_PATH = next((path for path in candidate_paths if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Improved 12-class dataset not found. Keep the GRU folder inside the main project folder.")

raw_df = pd.read_csv(DATA_PATH)
df = raw_df[["title_clean", "body_clean", "model_text", "target_genre", "use_for_training"]].dropna(subset=["model_text", "target_genre"]).copy()
df = df[df["use_for_training"].astype(bool)].copy()
df["model_text"] = df["model_text"].astype(str).str.replace(r"[\x00-\x1f\x7f-\x9f]", " ", regex=True).str.strip().str.replace(r"\s+", " ", regex=True)
df = df[df["model_text"] != ""].copy()
duplicates_removed = int(df.duplicated(subset=["model_text"]).sum())
df = df.drop_duplicates(subset=["model_text"], keep="first").reset_index(drop=True)

print("TensorFlow version:", tf.__version__)
print("Dataset path:", DATA_PATH)
print("Original rows:", len(raw_df))
print("Rows used:", len(df))
print("Duplicate model_text rows removed:", duplicates_removed)
print("Genres:", sorted(df["target_genre"].unique()))

display(df[["title_clean", "target_genre", "body_clean"]].head())
display(df["target_genre"].value_counts().rename_axis("genre").reset_index(name="rows"))

## 2. Split Dataset

In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df["target_genre"])
X = df["model_text"].astype(str).to_numpy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

print("Train rows:", len(X_train))
print("Validation rows:", len(X_val))
print("Test rows:", len(X_test))
print("Number of classes:", len(label_encoder.classes_))

## 3. Model Cell

In [ ]:
SEQUENCE_MAX_TOKENS = 12000
TFIDF_MAX_TOKENS = 20000
SEQUENCE_LENGTH = 220
EMBEDDING_DIM = 64
GRU_UNITS = 48
EPOCHS = 14
BATCH_SIZE = 32
PATIENCE = 3

sequence_vectorizer = layers.TextVectorization(
    max_tokens=SEQUENCE_MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
)
tfidf_vectorizer = layers.TextVectorization(
    max_tokens=TFIDF_MAX_TOKENS,
    output_mode="tf_idf",
    ngrams=2,
)
sequence_vectorizer.adapt(X_train)
tfidf_vectorizer.adapt(X_train)

num_classes = len(label_encoder.classes_)

inputs = layers.Input(shape=(), dtype=tf.string)

sequence_features = sequence_vectorizer(inputs)
sequence_features = layers.Embedding(SEQUENCE_MAX_TOKENS, EMBEDDING_DIM, mask_zero=True)(sequence_features)
sequence_features = layers.SpatialDropout1D(0.25)(sequence_features)
sequence_features = layers.Bidirectional(layers.GRU(
    GRU_UNITS,
    dropout=0.25,
    recurrent_dropout=0.10,
    kernel_regularizer=regularizers.l2(0.0005),
))(sequence_features)

tfidf_features = tfidf_vectorizer(inputs)
tfidf_features = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(0.0005))(tfidf_features)
tfidf_features = layers.Dropout(0.35)(tfidf_features)

combined_features = layers.Concatenate()([sequence_features, tfidf_features])
combined_features = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(0.0005))(combined_features)
combined_features = layers.Dropout(0.40)(combined_features)
outputs = layers.Dense(num_classes, activation="softmax")(combined_features)

gru_model = models.Model(inputs, outputs)
gru_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

gru_model.summary()

## 4. Training Cell With Epochs, Early Stopping, And Checkpoint

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=PATIENCE,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "genre_gru_checkpoint.keras",
        monitor="val_accuracy",
        save_best_only=True,
    ),
]

print("Training GRU")
print("Maximum epochs:", EPOCHS)
print("Early stopping patience:", PATIENCE)

history = gru_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=2,
)

best_val_accuracy_epoch = int(np.argmax(history.history["val_accuracy"]) + 1)
best_val_loss_epoch = int(np.argmin(history.history["val_loss"]) + 1)

print("Epochs completed:", len(history.history["loss"]))
print("Best validation accuracy epoch:", best_val_accuracy_epoch)
print("Best validation loss epoch:", best_val_loss_epoch)
print("Checkpoint saved: genre_gru_checkpoint.keras")

## 5. Results: Accuracy, Macro F1, Weighted F1, And Classification Report

In [ ]:
test_probabilities = gru_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
test_predictions = np.argmax(test_probabilities, axis=1)

val_loss, val_accuracy = gru_model.evaluate(X_val, y_val, batch_size=BATCH_SIZE, verbose=0)
test_accuracy = accuracy_score(y_test, test_predictions)
test_macro_f1 = f1_score(y_test, test_predictions, average="macro", zero_division=0)
test_weighted_f1 = f1_score(y_test, test_predictions, average="weighted", zero_division=0)

metrics = {
    "dataset": DATASET_NAME,
    "model": "Hybrid GRU",
    "validation_accuracy": float(val_accuracy),
    "test_accuracy": float(test_accuracy),
    "test_macro_f1": float(test_macro_f1),
    "test_weighted_f1": float(test_weighted_f1),
    "epochs_completed": int(len(history.history["loss"])),
    "best_validation_accuracy_epoch": int(np.argmax(history.history["val_accuracy"]) + 1),
}

print("Validation accuracy:", round(val_accuracy, 4))
print("Test accuracy:", round(test_accuracy, 4))
print("Test macro F1:", round(test_macro_f1, 4))
print("Test weighted F1:", round(test_weighted_f1, 4))
print()
print(classification_report(y_test, test_predictions, target_names=label_encoder.classes_, zero_division=0))

with open("gru_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)